# 03 · Masquage des compagnes

Anna Preto — AstroDeep Project

Avant / après masquage, ellipses des compagnes, remplissage par du ciel vide, et la décision à 0,02 sur l'écart d'ellipticité.

Lit les produits de la chaîne locale (`res config/local_demo.yaml all`). Les sorties de ce notebook montrent des pixels Rubin DP2 : **ne jamais les versionner** (effacer les sorties avant de commiter).

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from res.config import Config
from res import viz

CONFIG = Path("../config/local_demo.yaml")
cfg = Config.load(CONFIG)
cc = cfg["cuts"]
table = pd.read_parquet(cfg.data("30_processed", "table.parquet"))
plt.rcParams["figure.dpi"] = 110
print(f"{len(table):,} objets dans le catalogue")
from res.stages import stamps as st_mod
stamps, smeta = viz.load_stamps(cfg)
T = table.set_index("object_id")
R = table[table["in_region"]]
img = st_mod.load_images(cfg)
rubin_c, euclid_c = st_mod.companions(cfg, img, table)

In [ ]:
def companions_in(row, n=59):
    ox, oy = np.floor(row.x_tract + 0.5) - img["x0"] - n // 2, np.floor(row.y_tract + 0.5) - img["y0"] - n // 2
    rc = rubin_c[rubin_c["id"] != row.object_id]
    ec = euclid_c[~((euclid_c["ra"] == row.euclid_ra) & (euclid_c["dec"] == row.euclid_dec))]
    out = []
    for c, col in ((rc, "C1"), (ec, "C2")):
        sel = c["x"].between(ox - 20, ox + n + 20) & c["y"].between(oy - 20, oy + n + 20)
        for r in c[sel].itertuples():
            out.append((r.x - ox, r.y - oy, r.a, r.b, r.theta, col))
    return out

def before_after(rows, title_col):
    fig, axes = plt.subplots(3, len(rows), figsize=(2.3 * len(rows), 7.0), squeeze=False)
    for j, row in enumerate(rows.itertuples()):
        s = stamps[row.object_id]
        vmin, vmax = np.percentile(s["flux_raw"][3], [1, 99.5])
        viz.show(axes[0, j], s["flux_raw"][3], f"{getattr(row, title_col):.3f}", vmin=vmin, vmax=vmax)
        viz.show(axes[1, j], s["flux_raw"][3], vmin=vmin, vmax=vmax)
        viz.overlay(axes[1, j], s["companion_mask"], "C2", 0.45)
        viz.show(axes[2, j], s["flux"][3], vmin=vmin, vmax=vmax)
        xc, yc = viz.stamp_center(row)
        for ax in axes[:, j]:
            ax.set_xlim(-0.5, 58.5); ax.set_ylim(-0.5, 58.5)
            viz.draw_ellipse(ax, xc, yc, row.ellipse_a, row.ellipse_b, row.ellipse_theta, ec="C0", lw=0.9)
        for x, y, a, b, t, col in companions_in(row):
            viz.draw_ellipse(axes[0, j], x, y, a, b, t, ec=col, lw=0.7)
    axes[0, 0].set_ylabel("avant"); axes[1, 0].set_ylabel("masque"); axes[2, 0].set_ylabel("après")
    return fig

**Méthode** (`masking.method` dans la configuration). « footprint » s'appuie sur les empreintes de détection (plan `DETECTED`, dans au moins une bande) : l'empreinte qui contient la cible est gardée entière ; toute autre empreinte est remplacée par du ciel, élargie de `grow_px` pixels ; une compagne dont le centre tombe dans l'empreinte de la cible est masquée sur `shared_k` fois son ellipse ; une compagne non détectée (vue par Euclid seul) garde son ellipse. « ellipse » : la méthode de la recette, ellipse de demi-lumière seule, qui laissait les ailes des compagnes brillantes.

**Légende** — ligne 1 : avant, titre = grandeur de tri ; bleu : la cible ; orange : ellipses des compagnes Rubin ; vert : ellipses des compagnes Euclid. Ligne 2 : en vert, les pixels remplacés. Ligne 3 : après. Bande i, même échelle sur les trois lignes.

## Cibles gardées, masquage le plus étendu

In [ ]:
before_after(R[R['mask_pass'].astype(bool)].sort_values('masked_fraction', ascending=False).head(8), 'masked_fraction');

## Cibles écartées au masquage (|Δe| > 0,02)

In [ ]:
before_after(R[~R['mask_pass'].astype(bool)].sort_values('delta_e_mask').head(8), 'delta_e_mask');

In [ ]:
before_after(R[~R['mask_pass'].astype(bool)].sort_values('delta_e_mask', ascending=False).head(8), 'delta_e_mask');

## L'écart au masquage en chiffres

In [ ]:
d = R["delta_e_mask"]
print(f"mesures réussies avant et après : {d.notna().sum()} / {len(R)} ; écart nul (rien à masquer près de la "
      f"cible) : {(d == 0).sum()} ; > 0,02 : {(d > cc['delta_e_max']).sum()}")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.3))
axes[0].hist(np.log10(d[d > 0]), bins=40, color="0.4"); axes[0].axvline(np.log10(cc["delta_e_max"]), c="C3", ls="--")
axes[0].set_xlabel("log10 |Δe| (Δe > 0)")
axes[1].scatter(R["masked_fraction"], d, s=4, c="0.3"); axes[1].axhline(cc["delta_e_max"], c="C3", ls="--")
axes[1].set_yscale("symlog", linthresh=1e-3); axes[1].set_xlabel("fraction de pixels masqués"); axes[1].set_ylabel("|Δe|")
fail = ~R["shape_ok"].astype(bool)
bins = np.linspace(20, 28, 17)
h_all, _ = np.histogram(R["i_cModel_mag"], bins); h_f, _ = np.histogram(R.loc[fail, "i_cModel_mag"], bins)
axes[2].step(bins[:-1], np.where(h_all > 0, h_f / np.maximum(h_all, 1), np.nan), where="post", c="0.2")
axes[2].set_xlabel("i [mag]"); axes[2].set_ylabel("fraction de mesures HSM ratées")
plt.tight_layout()

## Le ciel de remplissage a-t-il le bruit du ciel ?

Pixels remplis contre pixels de ciel non détectés, bande i, normalisés par l'écart-type attendu.

In [ ]:
det = smeta["plane_bits"]["DETECTED"]
filled_bit = smeta["filled_bit"]
a, b = [], []
for oid in R["object_id"]:
    s = stamps[oid]
    sky_var = T.loc[oid, "sky_var_i"]
    m = s["maskbits"][3]
    a.append(s["flux"][3][(m & filled_bit) != 0] / np.sqrt(sky_var))
    b.append(s["flux_raw"][3][(m & (det | filled_bit)) == 0] / np.sqrt(sky_var))
a, b = np.concatenate(a), np.concatenate(b)
bins = np.linspace(-5, 5, 80)
plt.figure(figsize=(6, 3.3))
plt.hist(b, bins, density=True, histtype="step", color="0.4", label=f"ciel non détecté (σ={np.std(b[np.abs(b) < 5]):.2f})")
plt.hist(a, bins, density=True, histtype="step", color="C3", label=f"pixels remplis (σ={np.std(a[np.abs(a) < 5]):.2f})")
plt.yscale("log"); plt.xlabel("flux / σ attendu"); plt.legend(fontsize=8)